In [4]:
!mamba install pandas
import pandas as pd
import json
import sqlite3
import pandas as pd
pd.__version__


mambajs 0.19.13

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge, conda-forge

Solving environment...
Solving took 2.2332000000476837 seconds
All requested packages already installed.


'3.0.0'

In [5]:
import pandas as pd

orders_df = pd.read_csv("orders.csv")
orders_df.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


In [6]:
import json

with open("users.json", "r") as f:
    users_data = json.load(f)

users_df = pd.DataFrame(users_data)
users_df.head()



,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [7]:
import sqlite3


In [8]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
print("SQLite in-memory DB created")


SQLite in-memory DB created


In [9]:
with open("restaurants.sql", "r") as f:
    sql_script = f.read()

print(sql_script[:500])  # preview first 500 chars


CREATE TABLE restaurants (
restaurant_id INT,
restaurant_name TEXT,
cuisine TEXT,
rating FLOAT);

INSERT INTO restaurants VALUES (1, 'Restaurant_1', 'Chinese', 4.8);
INSERT INTO restaurants VALUES (2, 'Restaurant_2', 'Indian', 4.1);
INSERT INTO restaurants VALUES (3, 'Restaurant_3', 'Mexican', 4.3);
INSERT INTO restaurants VALUES (4, 'Restaurant_4', 'Chinese', 4.1);
INSERT INTO restaurants VALUES (5, 'Restaurant_5', 'Chinese', 4.8);
INSERT INTO restaurants VALUES (6, 'Restaurant_6', 'Chinese', 4


In [10]:
try:
    cursor.executescript(sql_script)
    print("SQL script executed successfully")
except Exception as e:
    print("SQL execution failed:", e)


SQL script executed successfully


In [11]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
cursor.fetchall()


[('restaurants',)]

In [12]:
cursor.execute("SELECT * FROM restaurants LIMIT 5;")
cursor.fetchall()


[(1, 'Restaurant_1', 'Chinese', 4.8),
 (2, 'Restaurant_2', 'Indian', 4.1),
 (3, 'Restaurant_3', 'Mexican', 4.3),
 (4, 'Restaurant_4', 'Chinese', 4.1),
 (5, 'Restaurant_5', 'Chinese', 4.8)]

In [13]:
import pandas as pd

restaurants_df = pd.read_sql_query(
    "SELECT * FROM restaurants",
    conn
)

restaurants_df.head()


,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


In [14]:
orders_users_df = orders_df.merge(
    users_df,
    on="user_id",
    how="left"
)

orders_users_df.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name,name,city,membership
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular


In [15]:
final_df = orders_users_df.merge(
    restaurants_df,
    on="restaurant_id",
    how="left"
)

final_df.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


In [16]:
final_df.to_csv("final_food_delivery_dataset.csv", index=False)


In [19]:
final_df[final_df["membership"]=="Gold"] \
.groupby("city")["total_amount"] \
.sum() \
.sort_values(ascending=False)


city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64

In [20]:
final_df.groupby("cuisine")["total_amount"].mean() \
.sort_values(ascending=False)


cuisine
Mexican    808.021344
Italian    799.448578
Indian     798.466011
Chinese    798.389020
Name: total_amount, dtype: float64

In [21]:
final_df.groupby("user_id")["total_amount"].sum() \
.gt(1000).sum()


np.int64(2544)

In [22]:
final_df["rating_range"] = pd.cut(
    final_df["rating"],
    bins=[3.0,3.5,4.0,4.5,5.0],
    labels=["3.0–3.5","3.6–4.0","4.1–4.5","4.6–5.0"]
)

final_df.groupby("rating_range")["total_amount"].sum()


rating_range
3.0–3.5    1881754.57
3.6–4.0    1717494.41
4.1–4.5    1960326.26
4.6–5.0    2197030.75
Name: total_amount, dtype: float64

In [23]:
final_df[final_df["membership"]=="Gold"] \
.groupby("city")["total_amount"].mean() \
.sort_values(ascending=False)


city
Chennai      808.459080
Hyderabad    806.421034
Bangalore    793.223756
Pune         781.162243
Name: total_amount, dtype: float64

In [24]:
final_df.groupby("cuisine").agg(
    restaurants=("restaurant_id","nunique"),
    revenue=("total_amount","sum")
).sort_values("restaurants")


,restaurants,revenue
cuisine,,
Chinese,120,1930504.65
Indian,126,1971412.58
Italian,126,2024203.80
Mexican,128,2085503.09


In [25]:
round(
    (final_df[final_df["membership"]=="Gold"].shape[0]/len(final_df)) * 100
)


50

In [29]:
[col for col in final_df.columns if "rest" in col.lower() or "name" in col.lower()]


['restaurant_id', 'restaurant_name_x', 'name', 'restaurant_name_y']

In [30]:
final_df.groupby(["restaurant_id", "name"]).agg(
    total_orders=("order_id", "count"),
    avg_order_value=("total_amount", "mean")
).query("total_orders < 20") \
 .sort_values("avg_order_value", ascending=False) \
 .head(1)


,,total_orders,avg_order_value
restaurant_id,name,,
75,User_1396,1,1499.83


In [32]:
final_df.groupby(["restaurant_id", "name"]).agg(
    total_orders=("order_id", "count"),
    avg_order_value=("total_amount", "mean")
).query("total_orders < 20") \
 .sort_values("avg_order_value", ascending=False) \
 .head(1)


,,total_orders,avg_order_value
restaurant_id,name,,
75,User_1396,1,1499.83


In [33]:
final_df[final_df["restaurant_id"] == 75]


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating,quarter,rating_range
1110,1111,2878,75,2023-04-14,1322.74,New Cafe South Indian,User_2878,Bangalore,Gold,Restaurant_75,Chinese,4.2,2023Q2,4.1–4.5
1146,1147,2013,75,2023-07-21,732.84,New Cafe South Indian,User_2013,Pune,Gold,Restaurant_75,Chinese,4.2,2023Q3,4.1–4.5
1358,1359,1146,75,2023-01-17,612.37,New Cafe South Indian,User_1146,Pune,Gold,Restaurant_75,Chinese,4.2,2023Q1,4.1–4.5
1451,1452,2474,75,2023-08-11,1481.64,New Cafe South Indian,User_2474,Chennai,Gold,Restaurant_75,Chinese,4.2,2023Q3,4.1–4.5
1739,1740,1961,75,2023-05-15,1185.07,New Cafe South Indian,User_1961,Pune,Regular,Restaurant_75,Chinese,4.2,2023Q2,4.1–4.5
2015,2016,2017,75,2023-01-03,112.31,New Cafe South Indian,User_2017,Chennai,Gold,Restaurant_75,Chinese,4.2,2023Q1,4.1–4.5
2763,2764,2298,75,2023-08-05,1036.72,New Cafe South Indian,User_2298,Bangalore,Regular,Restaurant_75,Chinese,4.2,2023Q3,4.1–4.5
2867,2868,368,75,2023-04-15,426.81,New Cafe South Indian,User_368,Pune,Regular,Restaurant_75,Chinese,4.2,2023Q2,4.1–4.5
2976,2977,2233,75,2023-10-05,548.98,New Cafe South Indian,User_2233,Hyderabad,Regular,Restaurant_75,Chinese,4.2,2023Q4,4.1–4.5
3347,3348,1503,75,2023-01-18,290.57,New Cafe South Indian,User_1503,Pune,Gold,Restaurant_75,Chinese,4.2,2023Q1,4.1–4.5


In [34]:
final_df.groupby(["membership","cuisine"])["total_amount"] \
.sum().sort_values(ascending=False)


membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

In [35]:
final_df.groupby("quarter")["total_amount"].sum() \
.sort_values(ascending=False)


quarter
2023Q3    2037385.10
2023Q4    2018263.66
2023Q1    1993425.14
2023Q2    1945348.72
2024Q1      17201.50
Freq: Q-DEC, Name: total_amount, dtype: float64

In [36]:
final_df[final_df["membership"] == "Gold"].shape[0]


4987

In [37]:
round(
    final_df[final_df["city"] == "Hyderabad"]["total_amount"].sum()
)


1889367

In [38]:
final_df["user_id"].nunique()


2883

In [39]:
round(
    final_df[final_df["membership"] == "Gold"]["total_amount"].mean(),
    2
)


np.float64(797.15)

In [40]:
final_df[final_df["rating"] >= 4.5].shape[0]


3374

In [41]:
final_df[final_df["rating"] >= 4.5].shape[0]


3374

In [42]:
top_city = (
    final_df[final_df["membership"] == "Gold"]
    .groupby("city")["total_amount"]
    .sum()
    .idxmax()
)

top_city

'Chennai'

In [44]:
gold_city_revenue = (
    final_df[final_df["membership"] == "Gold"]
    .groupby("city")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

gold_city_revenue


city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64

In [45]:
top_city = gold_city_revenue.index[0]

final_df[
    (final_df["membership"] == "Gold") &
    (final_df["city"] == top_city)
].shape[0]


1337